# Evaluating SIA outputs

Scores images produced by `training/infer_single.py` (or any edited images) against the source image **B**:

- **CLIP / SigLIP / BLIP**: how well an image matches a text description of the edit (higher is better).
- **LPIPS / DINOv2 / FaceID**: distance from B (lower means more of B is preserved).

Run `pip install -e .` from the repository root first so the `evaluation` package is importable.

In [ ]:
from PIL import Image

from evaluation import (
    BLIPEvaluator,
    CLIPEvaluator,
    DiNOFeatureDistanceEvaluator,
    FaceIdentityDistanceEvaluator,
    LPIPSFeatureDistanceEvaluator,
    SiglipEvaluator,
)

## 1) Inputs

Set the source image B, the images to score and the text describing the edit.

In [ ]:
image_b = Image.open("path/to/B.png").convert("RGB")

# label -> path of an output image to score
outputs = {
    "exlora (no suppression)": "path/to/output_none.png",
    "exlora (suppress e1)": "path/to/output_e1.png",
}
outputs = {label: Image.open(path).convert("RGB") for label, path in outputs.items()}

edit_text = "a person wearing sunglasses"

## 2) Load the evaluators

FaceID needs `insightface`; remove it if your images contain no faces.

In [ ]:
text_evaluators = {
    "CLIP": CLIPEvaluator(),
    "SigLIP": SiglipEvaluator(),
    "BLIP": BLIPEvaluator(),
}
distance_evaluators = {
    "LPIPS (vgg)": LPIPSFeatureDistanceEvaluator(net="vgg"),
    "DINOv2": DiNOFeatureDistanceEvaluator(),
    "FaceID": FaceIdentityDistanceEvaluator(),
}

## 3) Score

In [ ]:
for label, image in outputs.items():
    print(label)
    for name, evaluator in text_evaluators.items():
        print(f"  {name:12s} score    {evaluator.get_score(image, edit_text):.4f}")
    for name, evaluator in distance_evaluators.items():
        print(f"  {name:12s} distance {evaluator.get_distance(image_b, image):.4f}")